# Auditoría de soporte del wrapper `non_iid` en el split **train**

Este notebook cuenta, para el split **train real** (no `train + val`) de cada dataset/config con wrapper `non_iid`, cuántos ejemplos quedan sin soporte para ser sampleados por el wrapper porque no existe un cuadrilátero 2×2 completo que los incluya.

La lógica replica el pipeline de entrenamiento:

1. Carga la config de dataset.
2. Construye el dataset base de `data.training`.
3. Aplica el holdout OOD con `ood_validation_split`, cuando corresponde.
4. Aplica el `random_split` de `val_fraction` y conserva sólo `train`.
5. Usa los mismos atributos y flags relevantes del wrapper `non_iid` (`allowed_attributes`, `shared_other_attributes`, etc.).
6. Calcula cuántos elementos del `train` pueden participar en al menos un cuadrilátero 2×2 completo.

> Nota: el wrapper `non_iid` se usa para samplear grupos de 4 elementos; este notebook mide el soporte sobre el dataset subyacente de train antes de envolverlo, porque queremos saber qué ejemplos del split de train **nunca podrían aparecer** en un cuadrilátero válido.

## Parámetros

- `CONFIG_PATHS`: por defecto analiza todos los `configs/datasets/*_non_iid.yml`.
- `SEED`: debe coincidir con la seed del experimento; se usa tanto para resolver `${seed}` como para reproducir el split train/val.
- `FORCE_GENERAL_COMPOSITION_C1`: déjalo en `False` si las configs ya traen `split: general_composition` y `c: 1`. Cámbialo a `True` sólo si quieres simular explícitamente ese split sobre configs que todavía no lo tienen seteado.

In [ ]:
from pathlib import Path

# Analiza todas las configs non_iid disponibles. También puedes reemplazar esto por una lista explícita.
CONFIG_PATHS = sorted(Path("configs/datasets").glob("*_non_iid.yml"))

# Debe coincidir con la seed del run que quieres auditar.
SEED = 0

# Sólo activar si quieres forzar split=general_composition,c=1 aunque la config no lo indique.
FORCE_GENERAL_COMPOSITION_C1 = False
GENERAL_COMPOSITION_OVERRIDES = {
    "split": "general_composition",
    "c": 1,
}

CONFIG_PATHS

## Imports y helpers

Estos helpers reutilizan funciones del repo para que el cálculo sea consistente con el entrenamiento.

In [ ]:
from typing import Any, Dict

import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf
from torch.utils.data import random_split

from visgen.datasets import Cars3D, CLEVR, DSprites, IRAVEN, MPI3D, Shapes3D
from visgen.datasets import _config_attribute_names, _filter_allowed_attributes, _resolve_non_iid_cfg
from visgen.datasets.non_iid import _FourCaseSupportAnalyzer
from visgen.datasets.splits import training_subset_with_optional_ood_holdout
from visgen.utils.general import fix_random

DATASET_MAP = {
    "cars3d": Cars3D,
    "clevr": CLEVR,
    "dsprites": DSprites,
    "iraven": IRAVEN,
    "mpi3d": MPI3D,
    "shapes3d": Shapes3D,
}


def load_dataset_cfg(config_path: Path, seed: int = SEED):
    """Carga una config de dataset y resuelve `${seed}` como lo haría el run."""
    cfg = OmegaConf.merge({"seed": seed}, OmegaConf.load(config_path))
    if FORCE_GENERAL_COMPOSITION_C1:
        for key, value in GENERAL_COMPOSITION_OVERRIDES.items():
            cfg.data.training[key] = value
            if "testing" in cfg.data:
                cfg.data.testing[key] = value
    OmegaConf.resolve(cfg)
    return cfg


def non_iid_runtime_options(train_cfg, train_subset):
    """Replica las opciones relevantes que `_wrap_non_iid` pasa a `NonIIDWrapper`."""
    non_iid_cfg = _resolve_non_iid_cfg(train_cfg)
    if not non_iid_cfg or isinstance(non_iid_cfg, str):
        non_iid_cfg = {}

    allowed_attributes = non_iid_cfg.get("allowed_attributes")
    if not allowed_attributes:
        allowed_attributes = _config_attribute_names(train_cfg)
    allowed_attributes = _filter_allowed_attributes(train_subset, allowed_attributes)

    return {
        "allowed_attributes": allowed_attributes,
        "shared_other_attributes": non_iid_cfg.get("shared_other_attributes", True),
        "fully_iid": non_iid_cfg.get("fully_iid", False),
        "apply_to": list(non_iid_cfg.get("apply_to", [])) if non_iid_cfg.get("apply_to") else None,
        "deterministic": non_iid_cfg.get("deterministic", False),
        "max_resample_attempts": non_iid_cfg.get("max_resample_attempts", 10_000),
    }


def make_train_subset(cfg, seed: int = SEED):
    """Construye exactamente el split train usado por `get_dataloaders` para `data.training`."""
    fix_random(seed)
    train_cfg = cfg.data.training
    dataset_cls = DATASET_MAP[train_cfg.dataset]
    full_training_dataset = dataset_cls(**train_cfg)

    num_ood_val = train_cfg.num_ood_val if "num_ood_val" in train_cfg else 1
    train_before_val = training_subset_with_optional_ood_holdout(
        full_training_dataset,
        num_ood_val=num_ood_val,
    )

    val_fraction = float(train_cfg.val_fraction)
    val_size = int(val_fraction * len(train_before_val))
    train_size = len(train_before_val) - val_size
    split_generator = torch.Generator().manual_seed(seed)
    train_subset, val_subset = random_split(
        train_before_val,
        [train_size, val_size],
        generator=split_generator,
    )
    return full_training_dataset, train_before_val, train_subset, val_subset


def support_mask(train_subset, *, allowed_attributes, shared_other_attributes):
    analyzer = _FourCaseSupportAnalyzer(train_subset, allowed_attributes)
    return analyzer.valid_sample_mask(shared_other_attributes=shared_other_attributes), analyzer


def resolve_attr_name(analyzer, attr_idx: int) -> str:
    base_dataset, _ = analyzer._unwrap_subset(analyzer.dataset)
    if hasattr(base_dataset, "_attribute_indices"):
        reverse = {idx: name for name, idx in base_dataset._attribute_indices.items()}
        if hasattr(base_dataset, "_target_index_map"):
            reverse = {
                new_idx: reverse[orig_idx]
                for orig_idx, new_idx in base_dataset._target_index_map.items()
                if orig_idx in reverse
            }
        return reverse.get(attr_idx, f"attribute_{attr_idx}")
    return f"attribute_{attr_idx}"


def per_pair_support(train_subset, analyzer, *, shared_other_attributes):
    rows = []
    total = len(train_subset)
    for i, attr_a in enumerate(analyzer.attribute_indices):
        for attr_b in analyzer.attribute_indices[i + 1:]:
            if shared_other_attributes:
                mask = analyzer._mask_for_attr_pair_shared(attr_a, attr_b)
            else:
                mask = analyzer._mask_for_attr_pair_unshared(attr_a, attr_b)
            supported = int(mask.sum())
            rows.append({
                "attr_a": resolve_attr_name(analyzer, int(attr_a)),
                "attr_b": resolve_attr_name(analyzer, int(attr_b)),
                "supported": supported,
                "unsupported": total - supported,
                "support_ratio": supported / total if total else np.nan,
            })
    return pd.DataFrame(rows).sort_values(
        ["support_ratio", "supported", "attr_a", "attr_b"],
        ascending=[False, False, True, True],
    )


def analyze_config(config_path: Path, seed: int = SEED) -> Dict[str, Any]:
    cfg = load_dataset_cfg(config_path, seed=seed)
    full_ds, train_before_val, train_subset, val_subset = make_train_subset(cfg, seed=seed)
    opts = non_iid_runtime_options(cfg.data.training, train_subset)

    if opts["fully_iid"]:
        valid_mask = np.ones(len(train_subset), dtype=bool)
        analyzer = None
    else:
        valid_mask, analyzer = support_mask(
            train_subset,
            allowed_attributes=opts["allowed_attributes"],
            shared_other_attributes=opts["shared_other_attributes"],
        )

    total = len(train_subset)
    supported = int(valid_mask.sum())
    unsupported = total - supported
    result = {
        "config": str(config_path),
        "dataset": cfg.data.training.dataset,
        "split": cfg.data.training.split,
        "split_difficulty": cfg.data.training.get("split_difficulty"),
        "c": cfg.data.training.get("c"),
        "seed": seed,
        "train_before_val": len(train_before_val),
        "val_size": len(val_subset),
        "train_total": total,
        "supported": supported,
        "unsupported": unsupported,
        "support_ratio": supported / total if total else np.nan,
        "unsupported_ratio": unsupported / total if total else np.nan,
        "allowed_attributes": opts["allowed_attributes"],
        "shared_other_attributes": opts["shared_other_attributes"],
        "fully_iid": opts["fully_iid"],
        "non_iid_apply_to": opts["apply_to"],
        "analyzer": analyzer,
        "train_subset": train_subset,
    }
    return result

## Ejecutar análisis

Esta celda puede tardar y requiere que los datasets existan bajo las rutas declaradas en las configs (`data/...`). Si falta algún archivo de datos, se marca esa config como error sin detener el resto del análisis.

In [ ]:
results = []
errors = []

for config_path in CONFIG_PATHS:
    print(f"Analizando {config_path} ...")
    try:
        result = analyze_config(config_path, seed=SEED)
        results.append(result)
    except Exception as exc:
        errors.append({"config": str(config_path), "error": repr(exc)})
        print(f"  ERROR: {exc!r}")

summary_cols = [
    "config", "dataset", "split", "split_difficulty", "c", "seed",
    "train_before_val", "val_size", "train_total",
    "supported", "unsupported", "support_ratio", "unsupported_ratio",
    "allowed_attributes", "shared_other_attributes", "fully_iid", "non_iid_apply_to",
]
summary_df = pd.DataFrame([{k: v for k, v in row.items() if k in summary_cols} for row in results])
if not summary_df.empty:
    summary_df = summary_df[summary_cols].sort_values(["unsupported_ratio", "unsupported"], ascending=False)
summary_df

In [ ]:
# Errores de carga/configuración, típicamente por datasets faltantes en data/.
errors_df = pd.DataFrame(errors)
errors_df

## Lectura rápida: ejemplos no sampleables

La columna clave es `unsupported`: cantidad de elementos del split **train** que no pertenecen a ningún cuadrilátero 2×2 válido bajo la configuración del wrapper. `unsupported_ratio` es la misma cantidad normalizada por `train_total`.

In [ ]:
if not summary_df.empty:
    display(summary_df[[
        "dataset", "config", "split", "c", "train_total",
        "supported", "unsupported", "unsupported_ratio",
        "allowed_attributes", "shared_other_attributes",
    ]])

## Desglose por par de atributos

Este desglose muestra, para cada config, cuántos ejemplos tienen soporte si el cuadrilátero usa un par específico de atributos. El total final del resumen anterior usa la unión sobre todos los pares elegibles, igual que el wrapper puede samplear distintos pares.

In [ ]:
pair_tables = {}
for row in results:
    if row["fully_iid"] or row["analyzer"] is None:
        continue
    table = per_pair_support(
        row["train_subset"],
        row["analyzer"],
        shared_other_attributes=row["shared_other_attributes"],
    )
    pair_tables[row["config"]] = table
    print("\n===", row["config"], "===")
    display(table)


## Exportar resultados

Guarda CSVs en `out/non_iid_train_support/` para poder adjuntarlos a un run o compararlos entre seeds/configs.

In [ ]:
OUTPUT_DIR = Path("out/non_iid_train_support")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not summary_df.empty:
    summary_path = OUTPUT_DIR / f"summary_seed_{SEED}.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Resumen guardado en {summary_path}")

for config_path, table in pair_tables.items():
    safe_name = Path(config_path).stem
    pair_path = OUTPUT_DIR / f"{safe_name}_pairs_seed_{SEED}.csv"
    table.to_csv(pair_path, index=False)
    print(f"Desglose por pares guardado en {pair_path}")

if errors:
    errors_path = OUTPUT_DIR / f"errors_seed_{SEED}.csv"
    errors_df.to_csv(errors_path, index=False)
    print(f"Errores guardados en {errors_path}")